# 3D решение
Реализация рассеияния гауссова пакета на точечном потенциале Юкавы

## Импорты

In [2]:
import numpy as np

from scipy.sparse import diags, eye
from scipy.sparse.linalg import spsolve

import matplotlib.pyplot as plt
import matplotlib.animation as animation

## Уравнение пакета

Полное 3D уравнение Шрёдингера:
 $$i \frac{\partial \psi}{\partial t} = - \frac{1}{2} \left(\frac{\partial^2}{\partial x^2} + \frac{\partial^2}{\partial y^2} + \frac{\partial^2}{\partial z^2} \right) \psi + V \left( \sqrt{x^2 +y^2 +z^2} \right) \psi$$

- Летит вдоль $x$
- Смещен на $b$ по $y$ (прицельный параметр)
- По $z$ - симметричный гауссиан с центром $z=0$

## Потенциал и поглощающие границы

### Потенциал Юкавы

$$V_{\text{yk}}(r) = \frac{e^{-r/R}}{r + \varepsilon}$$

где:
- $R$ - характерный размер потенциала
- $\varepsilon = 0.5\,\Delta x$ (предотвращает сингулярность при $r \to 0$)

При $R \to \infty$ переходит в кулоновский $1/r$.

### Поглощающий слой

На границе введен потенциал $-iW(r)$:

$$W(r) = \begin{cases}
0, & r \leq R_{\max} - w, \\[4pt]
W_{0} \cdot \sin^2\left(\dfrac{\pi}{2} \cdot \dfrac{r - (R_{\max} - w)}{w}\right), & r > R_{\max} - w,
\end{cases}$$

где:
- $R_{\max} = L/2$ - граница области
- $w$ - ширина слоя
- $W_{0}$ - интенсивность

$\sin^2$ выбран для гладкости второй производной (минимизация отражения), мнимость для экспоненциального затухания.

### Итоговый потенциал

$$V(r) = V_{\text{yk}}(r) - i\,W(r)$$

In [6]:
# Параметры системы
L = 100.0  # длина системы
N = 99  # дискретизация
T = 8  # общее время симуляции

# Сетка
x = np.linspace(-L / 2, L / 2, N)
y = np.linspace(-L / 2, L / 2, N)
z = np.linspace(-L / 2, L / 2, N)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
R = np.sqrt(X**2 + Y**2 + Z**2)

hbar = 1.0  # пост. Дирака
m = 1.0  # масса

# Дифференциалы
dx = L/N
dt = 0.05

# Волновой пакет
sigma, x0, k0, b = 1.0, -35.0, 2.0, 5.0  # ширина, положение, импульс, прицельный
psi = np.exp(-((X - x0) ** 2 + (Y - b) ** 2 + Z ** 2)/ (4 * sigma ** 2)) * np.exp(1j * k0 * x)
psi /= np.sqrt(np.sum(np.abs(psi)**2) * dx**3) # нормировка

# Потенциал Юкавы
R_param = 1.0
eps = 0.5 * dx
V_yukawa = np.exp(-R/R_param)/(R+eps)

# Поглощающий слой
W_max = 2.0
width = 15.0 # ширина слоя
R_max = L/2
W = np.zeros_like(R)
mask = R > R_max - width
W[mask] = W_max * np.sin(np.pi/2 * (R[mask] - (R_max - width))/width)**2

# Итоговый потенциал
V_eff = V_yukawa - 1j * W